# Goalkeepers

### Import Libraries

In [3]:
from selenium import webdriver 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import pandas as pd
import requests
from lxml import html
from bs4 import BeautifulSoup


### Create web pages variables to display goalkeeper statistics by league

In [4]:
pl = "https://fbref.com/en/comps/9/keepers/Premier-League-Stats"
liga = "https://fbref.com/en/comps/12/keepers/La-Liga-Stats"
seriea = "https://fbref.com/en/comps/11/keepers/Serie-A-Stats"
bundesliga = "https://fbref.com/en/comps/20/keepers/Bundesliga-Stats"
ligue1 = "https://fbref.com/en/comps/13/keepers/Ligue-1-Stats"


### Create DataFrame

In [22]:
FBREFGK = pd.DataFrame(columns=['Player','Nation','Pos','Team', 'Age', 'Born','MP' ,'Starts','Min', '90s','GA', 'GA90', 'SoTA', 'Saves', 'Save%' ,'W', 'D','L','CS','CS%','PKatt','PKA','PKsv', 'PKm','Save%'])

### Defining path variable

In [24]:
brave_path = "C:/Program Files/BraveSoftware/Brave-Browser/Application/brave.exe" 

### gkscraper Function
Scrapes goalkeeper statistics from a league webpage URL and inserts the extracted data into a pandas DataFrame

In [26]:
def gkscraper (url):
    # First l'ets set up the web driver with the Brave browser
    options = Options()
    options.binary_location = brave_path
    driver = webdriver.Chrome(service=Service(), options=options)
    driver.get(url)

    # define a variable to hold the GKs table
    table = driver.find_element(By.XPATH , '//*[@id="stats_keeper"]/tbody')

    # Get all rows in the table
    rows = table.find_elements(By.TAG_NAME, 'tr')

    # Remove the header row
    rows.pop(25)
    # Create a list to hold infixed rows
    infixed_rows = []

    # Loop through each row and extract the data
    for row in rows:
        # transform the row data into a list
        row_data = row.text.split(' ')

        # Remove the first element (the rank)
        row_data.pop(0)  

        # Remove the 'Matches' element if it exists
        row_data.remove('Matches')

        # remove duplicate of nationality and fix player and team names
        ind = row_data.index('GK')
        row_data.pop(ind-2)
        ind = row_data.index('GK')
        if ind == 3:
            name = row_data.pop(0)+ ' ' + row_data.pop(0)
            row_data.insert(0, name)
        elif ind == 4:
            name = row_data.pop(0) + ' ' + row_data.pop(0) + ' ' + row_data.pop(0)
            row_data.insert(0, name)
        # Fixiing team names
        ind = row_data.index('GK')
        if len(row_data[ind+2]) > 2 :
            team = row_data.pop(ind+1) + ' ' + row_data.pop(ind+1)
            row_data.insert(ind+1, team)
        
        # Append the row data to the DataFrame
        if len(row_data) == 25 :
            FBREFGK.loc[len(FBREFGK)] = row_data
        elif len(row_data) == 24:
            row_data.append('N/A')
            FBREFGK.loc[len(FBREFGK)] = row_data
        else:
            infixed_rows.append(row_data)
        
    # Close the driver after scraping
    driver.quit()
    return FBREFGK

### Scraping phase

In [28]:
gkscraper(pl)
gkscraper(liga)
gkscraper(seriea)
gkscraper(bundesliga)
gkscraper(ligue1)

,Player,Nation,Pos,Team,Age,Born,MP,Starts,Min,90s,...,W,D,L,CS,CS%,PKatt,PKA,PKsv,PKm,Save%
0,Emil Audero,IDN,GK,Como,27,1997,8,8,720,8.0,...,2,2,4,0,0.0,4,4,0,0,0.0
1,Jean Butez,FRA,GK,Como,29,1995,19,18,"1,666",18.5,...,8,4,6,5,27.8,2,2,0,0,0.0
2,Elia Caprile,ITA,GK,Napoli,22,2001,4,3,325,3.6,...,3,0,0,3,100.0,0,0,0,0,N/A
3,Elia Caprile,ITA,GK,Cagliari,22,2001,18,18,"1,620",18.0,...,5,4,9,5,27.8,1,1,0,0,0.0
4,Marco Carnesecchi,ITA,GK,Atalanta,24,2000,34,34,"3,060",34.0,...,19,7,8,13,38.2,2,2,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,Matvei Safonov,RUS,GK,Paris S-G,25,1999,10,9,879,9.8,...,7,1,1,3,33.3,2,2,0,0,0.0
242,Brice Samba,FRA,GK,Rennes,30,1994,17,17,"1,529",17.0,...,7,0,10,5,29.4,3,1,2,0,66.7
243,Brice Samba,FRA,GK,Lens,30,1994,15,15,"1,350",15.0,...,6,6,3,7,46.7,4,4,0,0,0.0
244,Arnau Tenas,ESP,GK,Paris S-G,23,2001,1,1,90,1.0,...,1,0,0,0,0.0,0,0,0,0,N/A
